# AquaAlert — train MuRIL on HumAID (Colab)

Runtime → Change runtime type → **GPU** (T4 is enough; ~4 epochs, roughly 20-40 min).
This notebook runs the same code as `python -m aquaalert.classify.train` and writes
`models/muril/metrics.json` (per-class F1, macro-F1 over 7 and over the 5 large classes, bootstrap 95% CI,
for both the trained model and the keyword fallback).

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > GPU"
print(torch.cuda.get_device_name(0))

## 1. Get the code
Either clone your repo (set `REPO_URL`), or upload a zip of the project (excluding `.venv/`, `data/raw/`, `models/`).

In [ ]:
REPO_URL = ""  # e.g. "https://github.com/<user>/AquaAlert.git" (leave empty to upload a zip instead)

import os
if REPO_URL:
    !git clone {REPO_URL} AquaAlert
else:
    from google.colab import files
    up = files.upload()  # choose AquaAlert.zip
    !unzip -q -o {list(up)[0]} -d .
    if not os.path.exists("AquaAlert") : print("Zip had no AquaAlert/ folder; cd into the unzipped project below.")
%cd AquaAlert

In [ ]:
!pip install -q "transformers>=4.40" "datasets>=2.18" accelerate pydantic pandas scikit-learn python-dotenv huggingface_hub pyarrow

## 2. Download HumAID (HF mirror QCRI/HumAID-all)

In [ ]:
!python scripts/fetch_humaid.py

## 3. Train
Class-weighted cross-entropy (inverse class frequency), 4 epochs, checkpoint selected by dev macro-F1.

In [ ]:
!python -m aquaalert.classify.train \
    --train data/raw/humaid/train.csv --val data/raw/humaid/dev.csv --test data/raw/humaid/test.csv \
    --label-dataset humaid --out models/muril --epochs 4 --batch-size 32 --lr 3e-5 --max-length 96

## 4. Results

In [ ]:
import json
m = json.load(open("models/muril/metrics.json"))
for name, r in m["models"].items():
    print(name, {k: round(r[k], 4) for k in ("accuracy", "macro_f1_all7", "macro_f1_large5")}, r["ci95"])
print(m["training"]["dev_history"])

## 5. Save outputs
Download the model + metrics (or copy to Drive). Unzip into the project's `models/muril/` so the dashboard picks it up.

In [ ]:
!cd models && zip -qr muril.zip muril -x "muril/checkpoints/*"
from google.colab import files
files.download("models/muril.zip")